### Imports

In [2]:
import os
import string
import itertools
import numpy as np
import pandas as pd
from pathlib import Path

import sys
sys.path.append("..")

import torch
from cdt.metrics import SHD
from utils import custom_binary_metrics, estimate_with_PCMCI, run_inv_pcmci

# from simulation.simulation_tools import get_optimal_sim_XYP
# from CausalTime.tools import generate_CT
from simulation.simulation_baselines import (
    simulate_with_CT, 
    simulate_with_ACT, 
    simulate_with_SDV 
)

import warnings
warnings.filterwarnings("ignore")

COL_NAMES = list(string.ascii_uppercase) + ["".join(a) for a in list(itertools.permutations(list(string.ascii_uppercase), r=2))]

Detecting 1 CUDA device(s).


 See https://github.com/google-research/timesfm/blob/master/README.md for updated APIs.
Loaded PyTorch TimesFM, likely because python version is 3.10.9 (tags/v3.10.9:1dd9be6, Dec  6 2022, 20:01:21) [MSC v.1934 64 bit (AMD64)].


### Simulation

In [9]:
par_dir = Path(os.getcwd()).parents[1].as_posix() 
save_dir = f"{par_dir}/data/results/cd_efficacy"

# Data structure is as such for convenient comparison with CausalTime
def get_data_dict(
        FN: str, 
        par_dir=Path(os.getcwd()).parents[1].as_posix()
):
    if FN in ['AirQualityUCI', 'air_quality_mini', 'bike-usage', 'ETTh1', 'ETTm1', 'outdoor', 'WTH']:
        return {
            filename.split(".csv")[0]: {
                'data_path': f"{par_dir}/data/MvTS/{FN}/",
                'data_type': 'fmri',
                'task': filename, 
                'straight_path': f"{par_dir}/data/MvTS/{FN}/" + f"{filename}"
            } for filename in os.listdir(f"{par_dir}/data/MvTS/{FN}/")}
    elif FN=="cp_style":
        return {
            filename.split(".csv")[0]: {
                'data_path': f"{par_dir}/data/{FN}/increasing_edges_cp_1/data",
                'data_type': 'fmri',
                'task': filename, 
                'straight_path': f"{par_dir}/data/{FN}/increasing_edges_cp_1/data/" + f"{filename}"
            } for filename in os.listdir(f"{par_dir}/data/{FN}/increasing_edges_cp_1/data")}
    elif FN=="fMRI":
        return {
            filename.split(".csv")[0]: {
                'data_path': f"{par_dir}/data/{FN}/timeseries/",
                'data_type': 'fmri',
                'task': filename, 
                'straight_path': f"{par_dir}/data/{FN}/timeseries/" + f"{filename}"
            } for filename in os.listdir(f"{par_dir}/data/{FN}/timeseries")}
    elif FN=="finance":
        return {
            filename.split(".csv")[0]: {
                'data_path': f"{par_dir}/data/{FN}/",
                'data_type': 'fmri',
                'task': filename, 
                'straight_path': f"{par_dir}/data/{FN}/" + f"{filename}"
            } for filename in os.listdir(f"{par_dir}/data/{FN}")}

# Data loop
for FN in ['AirQualityUCI', 'air_quality_mini', 'bike-usage', 
        #    'ETTh1', 
        #    'ETTm1', 
           'outdoor', 'WTH', 'fMRI', 'finance', 
        #    'cp_style'
           ]:

    DATA_DICT = get_data_dict(FN=FN)

    # CausalTime Parameters
    PARAMS = {
        "batch_size" : 32, 
        "hidden_size" : 128, 
        "num_layers" : 2, 
        "dropout" : 0.1, 
        "seq_length" : 20, 
        "test_size" : 0.2, 
        "learning_rate" : 0.0001, 
        "n_epochs" : 1, 
        "flow_length" : 4, 
        "gen_n" : 20, 
        "n" : 2000,
        "arch_type" : "MLP", 
        "save_path" : "outputs/", 
        "log_dir" : "log/", 
    }

    for k, v in list(DATA_DICT.items())[:]:

        try:
        
            # info
            filename = v['task']
            print(f" \n------------- {filename} ---------------\n ")

            # data
            true_data = pd.read_csv(v["straight_path"])
            true_data = true_data.rename(columns=dict(zip(true_data.columns, COL_NAMES[:true_data.shape[1]])))
            
            # adjust timesteps for computation time 
            print(f"true data length: {true_data.shape[0]}")

            # shorten true data
            if true_data.shape[0]>1000:
                anchor = np.random.uniform(low=0, high=true_data.shape[0]-1000)
                true_data = true_data.loc[anchor : anchor + 1000, :]
                print(f"true data length (adjusted): {true_data.shape[0]}")

            # \epsilon added to avoid computation errors w/ PCMCI
            for i in range(true_data.shape[0]):
                for j in range(true_data.shape[1]):
                    if true_data.iloc[i, j] == 0:
                        true_data.iloc[i, j] += np.random.uniform(low=0.0001, high=0.001)
            

            """ ____________________________________ Simulate w/ ACT ____________________________________ """

            tcs_data = simulate_with_ACT(true_data=true_data)


            # print("""\n ____________________________________ Simulate w/ CausalTime ____________________________________ \n""")

            # true_pd, pro_true_pd, skimmed_pd, pro_gen_pd = generate_CT(
            #         batch_size=PARAMS["batch_size"], 
            #         hidden_size=PARAMS["hidden_size"], 
            #         num_layers=PARAMS["num_layers"], 
            #         dropout=PARAMS["dropout"], 
            #         seq_length=PARAMS["seq_length"], 
            #         test_size=PARAMS["test_size"], 
            #         learning_rate=PARAMS["learning_rate"], 
            #         n_epochs=PARAMS["n_epochs"], 
            #         flow_length=PARAMS["flow_length"], 
            #         gen_n=PARAMS["gen_n"], 
            #         n=PARAMS["n"],
            #         arch_type=PARAMS["arch_type"], 
            #         save_path=PARAMS["save_path"], 
            #         log_dir=PARAMS["log_dir"], 
            #         data_path=v["data_path"],
            #         data_type= v["data_type"], 
            #         task= v["task"],
            #     )
            # ct_data = pro_gen_pd.copy()


            """ ____________________________________ Simulate w/ SDV ____________________________________ """

            # tcs_data = simulate_with_SDV(true_data=true_data)


            # Store
            os.makedirs(f"{save_dir}/simulated_tcs/{FN}/", exist_ok=True)
            tcs_data.to_csv(f"{save_dir}/simulated_tcs/{FN}/{filename}", index=False)
            # os.makedirs(f"{save_dir}/simulated_ct/{FN}/", exist_ok=True)
            # ct_data.to_csv(f"{save_dir}/simulated_ct/{FN}/{filename}", index=False)
            # os.makedirs(f"{save_dir}/simulated_sdv/{FN}/", exist_ok=True)
            # tcs_data.to_csv(f"{save_dir}/simulated_sdv/{FN}/{filename}", index=False)
            
            
        except:
            print(f"LOG: CD Efficacy: Error occured when simulating from {FN}.")
            continue

 
------------- AirQualityUCI_boot_0.csv ---------------
 
true data length: 2000
true data length (adjusted): 1000
LOG: Optimal Simulation: 12 TCS configurations are to be tested ...
LOG: Optimal Simulation: Enforcing sparcity penalty ...
LOG: CD Efficacy: Error occured when simulating from AirQualityUCI.
 
------------- AirQualityUCI_boot_1.csv ---------------
 
true data length: 2000
true data length (adjusted): 1000
LOG: Optimal Simulation: 12 TCS configurations are to be tested ...
LOG: Optimal Simulation: Enforcing sparcity penalty ...
LOG: CD Efficacy: Error occured when simulating from AirQualityUCI.
 
------------- AirQualityUCI_boot_2.csv ---------------
 
true data length: 2000
true data length (adjusted): 1000
LOG: Optimal Simulation: 12 TCS configurations are to be tested ...
LOG: Optimal Simulation: Enforcing sparcity penalty ...
LOG: CD Efficacy: Error occured when simulating from AirQualityUCI.
 
------------- AirQualityUCI_boot_3.csv ---------------
 
true data length:

100%|██████████| 100/100 [00:00<00:00, 613.50it/s]


LOG: Optimal Simulation: Sparsity Penalty: eq. found: idx=2 | auc=0.9989254599857831 | edges=49.0


100%|██████████| 100/100 [00:00<00:00, 800.02it/s]


LOG: Optimal Simulation: Sparsity Penalty: eq. found: idx=3 | auc=0.9972557901175382 | edges=49.0


...

LOG: Optimal Simulation: Sparsity Penalty: eq. found: idx=9 | auc=1.0 | edges=82.0


100%|██████████| 100/100 [00:00<00:00, 713.68it/s]


LOG: Optimal Simulation: Sparsity Penalty: eq. found: idx=10 | auc=1.0 | edges=82.0


100%|██████████| 100/100 [00:00<00:00, 709.22it/s]


LOG: Optimal Simulation: Sparsity Penalty: eq. found: idx=11 | auc=1.0 | edges=82.0
LOG: Optimal Simulation: Sparsity Penalty: opti case: idx=0 | auc=1.0 | edges=42.0


### CD Efficacy

In [3]:
ori_paths = {
    'air_quality_mini' : Path(os.getcwd()).parents[1] / "data" / "MvTS" / "air_quality_mini",
    'AirQualityUCI' : Path(os.getcwd()).parents[1] / "data" / "MvTS" / "AirQualityUCI",
    'bike-usage' : Path(os.getcwd()).parents[1] / "data" / "MvTS" / "bike-usage",
    'cp_style' : Path(os.getcwd()).parents[1] / "data" / "cp_style" / "increasing_edges_cp_1" / "data",
    'outdoor' : Path(os.getcwd()).parents[1] / "data" / "MvTS" / "outdoor",
    'ETTh1' : Path(os.getcwd()).parents[1] / "data" / "MvTS" / "ETTh1",
    'ETTm1' : Path(os.getcwd()).parents[1] / "data" / "MvTS" / "ETTm1",
    'fMRI' : Path(os.getcwd()).parents[1] / "data" / "fMRI" / "timeseries",
    'WTH' : Path(os.getcwd()).parents[1] / "data" / "MvTS" / "WTH",
    'finance' : Path(os.getcwd()).parents[1] / "data" / "finance",
}

sim_path = Path(os.getcwd()).parents[1] / "data" / "results" / "cd_efficacy"

res_ct = {}
res_tcs = {}
res_sdv = {}
res_both = {}

for FN in ori_paths.keys():

    res_ct[FN] = {}
    res_tcs[FN] = {}
    res_sdv[FN] = {}
    res_both[FN] = {}

    print(FN)
    for k in os.listdir(ori_paths[FN]):
        try:
            true_data = pd.read_csv(ori_paths[FN] / k)
            true_data = true_data.rename(columns=dict(zip(true_data.columns, COL_NAMES[:true_data.shape[1]])))
            tcs_data = pd.read_csv(sim_path / "simulated_tcs" / FN / k)
            sdv_data = pd.read_csv(sim_path / "simulated_sdv" / FN / k)
            ct_data = pd.read_csv(sim_path / "simulated_ct" / FN / k)
            print(f"LOG: file was read successfully for all methods! Shapes: tcs : {tcs_data.shape}, sdv : {sdv_data.shape}, ct : {ct_data.shape}")

            # Fix potential length mismatches
            # assert ct_data.shape == tcs_data.shape, AssertionError("Different data shape for TCS and CausalTime.")
            min_len = min([true_data.shape[0], tcs_data.shape[0], sdv_data.shape[0], ct_data.shape[0]])
            true_data = true_data[:min_len]
            tcs_data = tcs_data[:min_len]
            sdv_data = sdv_data[:min_len]
            ct_data = ct_data[:min_len]

            print(f"- {k}")

            # adj_cp_true, _ = estimate_with_PCMCI(true_data=true_data)
            # adj_cp_tcs, _ = estimate_with_PCMCI(true_data=tcs_data)
            # adj_cp_sdv, _ = estimate_with_PCMCI(true_data=sdv_data)
            # adj_cp_ct, _ = estimate_with_PCMCI(true_data=ct_data)
            adj_cp_true = run_inv_pcmci(sample=true_data)
            adj_cp_tcs = run_inv_pcmci(sample=tcs_data)
            adj_cp_sdv = run_inv_pcmci(sample=sdv_data)
            adj_cp_ct = run_inv_pcmci(sample=ct_data)

            tpr, fpr, tnr, fnr, auc = custom_binary_metrics(torch.tensor(adj_cp_tcs), torch.tensor(adj_cp_true), verbose=False)
            # shd = SHD(target=adj_cp_true.numpy(), pred=adj_cp_tcs.numpy())
            res_tcs[FN][k] = auc.item()
            # res_tcs[FN][k] = shd

            tpr, fpr, tnr, fnr, auc = custom_binary_metrics(torch.tensor(adj_cp_sdv), torch.tensor(adj_cp_true), verbose=False)
            # shd = SHD(target=adj_cp_true.numpy(), pred=adj_cp_sdv.numpy())
            res_sdv[FN][k] = auc.item()
            # res_sdv[FN][k] = shd

            tpr, fpr, tnr, fnr, auc = custom_binary_metrics(torch.tensor(adj_cp_ct), torch.tensor(adj_cp_true), verbose=False)
            # shd = SHD(target=adj_cp_true.numpy(), pred=adj_cp_ct.numpy())
            res_ct[FN][k] = auc.item()
            # res_ct[FN][k] = shd

        except Exception as e:
            print(f"LOG: {e}")
            continue
    
    res_both[FN]["TCS_mean"] = np.array(list(res_tcs[FN].values())).mean().round(2)
    res_both[FN]["SDV_mean"] = np.array(list(res_sdv[FN].values())).mean().round(2)
    res_both[FN]["CT_mean"] = np.array(list(res_ct[FN].values())).mean().round(2)

    res_both[FN]["TCS_var"] = np.array(list(res_tcs[FN].values())).var().round(2)
    res_both[FN]["SDV_var"] = np.array(list(res_sdv[FN].values())).var().round(2)
    res_both[FN]["CT_var"] = np.array(list(res_ct[FN].values())).var().round(2)
    
pd.DataFrame(data=res_both).T

air_quality_mini
LOG: file was read successfully for all methods! Shapes: tcs : (1000, 36), sdv : (1001, 36), ct : (1500, 36)
- air_quality_mini_boot_0.csv
LOG: file was read successfully for all methods! Shapes: tcs : (1000, 36), sdv : (1001, 36), ct : (1500, 36)
- air_quality_mini_boot_1.csv
LOG: file was read successfully for all methods! Shapes: tcs : (1000, 36), sdv : (1001, 36), ct : (1500, 36)
- air_quality_mini_boot_2.csv
LOG: file was read successfully for all methods! Shapes: tcs : (1000, 36), sdv : (1001, 36), ct : (1500, 36)
- air_quality_mini_boot_3.csv
LOG: file was read successfully for all methods! Shapes: tcs : (1000, 36), sdv : (1001, 36), ct : (1500, 36)
- air_quality_mini_boot_4.csv
AirQualityUCI
LOG: [Errno 2] No such file or directory: 'c:\\Users\\skypl\\Documents\\GitHub\\TCS\\data\\results\\cd_efficacy\\simulated_tcs\\AirQualityUCI\\AirQualityUCI_boot_0.csv'
LOG: [Errno 2] No such file or directory: 'c:\\Users\\skypl\\Documents\\GitHub\\TCS\\data\\results\\cd_ef

,TCS_mean,SDV_mean,CT_mean,TCS_var,SDV_var,CT_var
air_quality_mini,0.60,0.51,0.58,0.00,0.00,0.00
AirQualityUCI,0.62,0.60,0.76,0.00,0.00,0.00
bike-usage,0.63,0.58,0.65,0.02,0.00,0.01
cp_style,0.94,0.87,0.87,0.02,0.01,0.00
outdoor,0.90,0.88,0.48,0.00,0.00,0.00
ETTh1,0.76,0.71,0.68,0.00,0.00,0.00
ETTm1,0.71,0.67,0.67,0.01,0.01,0.00
fMRI,0.75,0.67,0.77,0.01,0.00,0.01
WTH,0.64,0.59,0.57,0.00,0.00,0.00
finance,0.73,0.50,0.67,0.00,0.00,0.00


In [4]:
display(pd.DataFrame(data=res_both).T.loc[["air_quality_mini", "AirQualityUCI", "bike-usage", "outdoor", "ETTh1", "ETTm1", "WTH"]].mean())
display(pd.DataFrame(data=res_both).T.loc[["fMRI", "finance"]].mean())
display(pd.DataFrame(data=res_both).T.loc[["cp_style"]].mean())

TCS_mean    0.694286
SDV_mean    0.648571
CT_mean     0.627143
TCS_var     0.004286
SDV_var     0.001429
CT_var      0.001429
dtype: float64

TCS_mean    0.740
SDV_mean    0.585
CT_mean     0.720
TCS_var     0.005
SDV_var     0.000
CT_var      0.005
dtype: float64

TCS_mean    0.94
SDV_mean    0.87
CT_mean     0.87
TCS_var     0.02
SDV_var     0.01
CT_var      0.00
dtype: float64